# Memoria Conversacional: Gestión de Estado en LCEL

## Objetivo

Añadir gestión de estado (memoria) a nuestras cadenas LCEL para permitir conversaciones naturales con preguntas de seguimiento como "¿Y qué más?", "¿Me lo resumes?" o "¿Cómo me llamo?".

### ¿Por qué necesitamos memoria?

Los modelos de lenguaje son **stateless** (sin estado). Esto significa que cada llamada es independiente y el modelo no recuerda conversaciones anteriores a menos que se lo proporcionemos explícitamente.

En este notebook aprenderás a:
- Entender por qué los LLMs son stateless
- Implementar memoria conversacional con LangChain
- Crear cadenas LCEL que mantienen el contexto
- Simular conversaciones con seguimiento


In [ ]:
# Setup inicial
import sys
import os

# Hack para importar desde src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.models import get_local_llm

print("✓ Imports completados")


## Teoría del Estado: ¿Por qué los LLMs son Stateless?

### El Problema

Los modelos de lenguaje (LLMs) son **stateless** por diseño:

1. **Cada llamada es independiente**: El modelo no mantiene información entre invocaciones
2. **Sin memoria interna**: No hay un "estado" que persista entre requests
3. **Input completo requerido**: Necesitas pasar todo el contexto en cada llamada

### Ejemplo del Problema

Sin memoria:
```
Usuario: "Me llamo Juan"
IA: "Hola Juan, mucho gusto"
Usuario: "¿Cómo me llamo?"
IA: "No tengo esa información"  ❌ (Olvidó el nombre)
```

Con memoria:
```
Usuario: "Me llamo Juan"
IA: "Hola Juan, mucho gusto"
Usuario: "¿Cómo me llamo?"
IA: "Te llamas Juan"  ✅ (Recuerda el nombre)
```

### La Solución: Historial de Mensajes

Necesitamos mantener un **historial de mensajes** y pasarlo en cada llamada junto con el mensaje actual.


In [ ]:
# Demostración: Sin memoria (stateless)
from langchain_core.messages import HumanMessage, AIMessage

llm = get_local_llm()

# Primera conversación
mensaje1 = HumanMessage(content="Me llamo Juan")
respuesta1 = llm.invoke([mensaje1])
print("Usuario: Me llamo Juan")
print(f"IA: {respuesta1.content}\n")

# Segunda conversación (sin contexto previo)
mensaje2 = HumanMessage(content="¿Cómo me llamo?")
respuesta2 = llm.invoke([mensaje2])
print("Usuario: ¿Cómo me llamo?")
print(f"IA: {respuesta2.content}")
print("\n❌ La IA no recuerda el nombre porque no tiene memoria")


## Componentes de Memoria en LangChain

LangChain proporciona componentes para gestionar memoria:

1. **`ChatMessageHistory`**: Almacena el historial de mensajes de una conversación
2. **`RunnableWithMessageHistory`**: Envuelve una cadena para añadir automáticamente el historial
3. **`MessagesPlaceholder`**: Placeholder en el prompt para inyectar el historial

### Flujo con Memoria:

```
Usuario envía mensaje
    ↓
RunnableWithMessageHistory obtiene historial
    ↓
Historial + Mensaje actual → Prompt
    ↓
LLM genera respuesta
    ↓
Respuesta se guarda en historial
```


In [ ]:
# Importar componentes de memoria
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

print("✓ Componentes de memoria importados")


## Paso 1: Prompt con Historial

El prompt debe incluir un `MessagesPlaceholder` para el historial. Esto permite que LangChain inyecte automáticamente los mensajes anteriores en el prompt.


In [ ]:
# Crear prompt con placeholder para el historial
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente amigable y conversacional. Responde de forma natural y recuerda el contexto de la conversación."),
    MessagesPlaceholder(variable_name="history"),  # Placeholder para el historial
    ("human", "{input}")  # Mensaje actual del usuario
])

print("✓ Prompt creado con MessagesPlaceholder")
print("\nEstructura del prompt:")
print("  1. System message (instrucciones)")
print("  2. History (mensajes anteriores - se inyecta automáticamente)")
print("  3. Human message (input actual)")


### ¿Qué es MessagesPlaceholder?

`MessagesPlaceholder(variable_name="history")` es un marcador de posición que le dice a LangChain:

- "Aquí va el historial de mensajes"
- El historial se inyecta automáticamente cuando se ejecuta la cadena
- Los mensajes se formatean correctamente para el modelo

Sin este placeholder, el modelo no recibiría el historial y no podría recordar conversaciones anteriores.


## Paso 2: Crear la Cadena Base

Primero creamos una cadena simple sin memoria. Luego la envolveremos con `RunnableWithMessageHistory`.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Instanciar el modelo
llm = get_local_llm()

# Crear la cadena base (sin memoria aún)
chain = prompt | llm | StrOutputParser()

print("✓ Cadena base creada")
print("  Estructura: prompt → llm → parser")


## Paso 3: Gestión de Historiales

Necesitamos una forma de almacenar y recuperar historiales de conversación. Usaremos un diccionario en memoria (en producción, usarías una base de datos).


In [ ]:
# Almacenar historiales en memoria (diccionario)
# En producción, esto estaría en una base de datos
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """
    Obtiene o crea el historial de mensajes para una sesión.
    
    Args:
        session_id: Identificador único de la sesión/conversación
        
    Returns:
        ChatMessageHistory para la sesión
    """
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

print("✓ Función de gestión de historiales creada")
print("  Cada sesión tiene su propio historial independiente")


### Explicación de get_session_history

Esta función:
- Recibe un `session_id` (identificador único de la conversación)
- Si la sesión no existe, crea un nuevo `ChatMessageHistory` vacío
- Si la sesión existe, devuelve el historial existente
- Permite múltiples conversaciones simultáneas (cada una con su propio historial)

En producción, reemplazarías el diccionario `store` con una base de datos (Redis, PostgreSQL, etc.).


## Paso 4: Envolver la Cadena con Memoria

Ahora envolvemos la cadena base con `RunnableWithMessageHistory`. Esto añade automáticamente:
- Obtención del historial antes de ejecutar
- Inyección del historial en el prompt
- Guardado de la respuesta en el historial


In [ ]:
# Envolver la cadena con memoria
chain_with_memory = RunnableWithMessageHistory(
    chain,  # Cadena base
    get_session_history,  # Función para obtener historial
    input_messages_key="input",  # Clave del input en el prompt
    history_messages_key="history",  # Clave del historial en el prompt
)

print("✓ Cadena con memoria creada")
print("\nConfiguración:")
print(f"  - Input key: 'input'")
print(f"  - History key: 'history'")
print(f"  - Session management: get_session_history")


### Parámetros de RunnableWithMessageHistory

- **`chain`**: La cadena base a envolver
- **`get_session_history`**: Función que obtiene el historial para una sesión
- **`input_messages_key`**: Nombre de la variable en el prompt para el input actual
- **`history_messages_key`**: Nombre de la variable en el prompt para el historial (debe coincidir con `MessagesPlaceholder`)


## Paso 5: Simulación de Chat

Ahora simularemos una conversación donde la IA recuerda el contexto. Usaremos un `session_id` para identificar la conversación.


In [ ]:
# Simulación de conversación con memoria
session_id = "conversacion_1"

print("=" * 60)
print("SIMULACIÓN DE CHAT CON MEMORIA")
print("=" * 60)

# Turno 1: Usuario se presenta
print("\n[TURNO 1]")
input1 = "Me llamo Juan y soy desarrollador de software"
print(f"Usuario: {input1}")

config = {"configurable": {"session_id": session_id}}
respuesta1 = chain_with_memory.invoke(
    {"input": input1},
    config=config
)
print(f"IA: {respuesta1}")

# Verificar que el historial se guardó
historial = get_session_history(session_id)
print(f"\n✓ Historial guardado: {len(historial.messages)} mensajes")


In [ ]:
# Turno 2: Pregunta sobre información previa
print("\n[TURNO 2]")
input2 = "¿Cómo me llamo?"
print(f"Usuario: {input2}")

respuesta2 = chain_with_memory.invoke(
    {"input": input2},
    config=config
)
print(f"IA: {respuesta2}")

# Verificar historial
historial = get_session_history(session_id)
print(f"\n✓ Historial actualizado: {len(historial.messages)} mensajes")
print("  ✅ La IA recuerda el nombre porque tiene acceso al historial")


In [ ]:
# Turno 3: Pregunta de seguimiento
print("\n[TURNO 3]")
input3 = "¿Y a qué me dedico?"
print(f"Usuario: {input3}")

respuesta3 = chain_with_memory.invoke(
    {"input": input3},
    config=config
)
print(f"IA: {respuesta3}")

# Mostrar el historial completo
historial = get_session_history(session_id)
print(f"\n✓ Historial completo ({len(historial.messages)} mensajes):")
for i, msg in enumerate(historial.messages, 1):
    tipo = "Usuario" if msg.__class__.__name__ == "HumanMessage" else "IA"
    contenido = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
    print(f"  {i}. {tipo}: {contenido}")


## Ejemplo: Múltiples Conversaciones

Cada `session_id` mantiene su propio historial independiente. Esto permite múltiples conversaciones simultáneas.


In [ ]:
# Conversación 1
print("=" * 60)
print("CONVERSACIÓN 1 (session_id: 'user_alice')")
print("=" * 60)

session_alice = "user_alice"
config_alice = {"configurable": {"session_id": session_alice}}

resp1 = chain_with_memory.invoke(
    {"input": "Mi nombre es Alice y me gusta la programación en Python"},
    config=config_alice
)
print(f"Usuario: Mi nombre es Alice y me gusta la programación en Python")
print(f"IA: {resp1}\n")

resp2 = chain_with_memory.invoke(
    {"input": "¿Qué me gusta hacer?"},
    config=config_alice
)
print(f"Usuario: ¿Qué me gusta hacer?")
print(f"IA: {resp2}\n")

# Conversación 2 (independiente)
print("=" * 60)
print("CONVERSACIÓN 2 (session_id: 'user_bob')")
print("=" * 60)

session_bob = "user_bob"
config_bob = {"configurable": {"session_id": session_bob}}

resp3 = chain_with_memory.invoke(
    {"input": "Soy Bob y trabajo como diseñador gráfico"},
    config=config_bob
)
print(f"Usuario: Soy Bob y trabajo como diseñador gráfico")
print(f"IA: {resp3}\n")

resp4 = chain_with_memory.invoke(
    {"input": "¿Cuál es mi profesión?"},
    config=config_bob
)
print(f"Usuario: ¿Cuál es mi profesión?")
print(f"IA: {resp4}\n")

print("✓ Cada conversación mantiene su propio historial independiente")


In [ ]:
# Simulación de chat con múltiples turnos
session_chat = "chat_interactivo"
config_chat = {"configurable": {"session_id": session_chat}}

# Lista de mensajes simulados
mensajes_simulados = [
    "Hola, soy María",
    "¿Cómo me llamo?",
    "Tengo 25 años",
    "¿Cuántos años tengo?",
    "Me gusta leer libros de ciencia ficción",
    "¿Qué tipo de libros me gustan?"
]

print("=" * 60)
print("CHAT SIMULADO CON MEMORIA")
print("=" * 60)

for i, mensaje in enumerate(mensajes_simulados, 1):
    print(f"\n[TURNO {i}]")
    print(f"Usuario: {mensaje}")
    
    respuesta = chain_with_memory.invoke(
        {"input": mensaje},
        config=config_chat
    )
    print(f"IA: {respuesta}")

print("\n" + "=" * 60)
print("✓ La IA mantiene el contexto a lo largo de toda la conversación")


## Resumen: Componentes de Memoria

### Flujo Completo:

1. **Usuario envía mensaje** → `{"input": "mensaje"}`
2. **RunnableWithMessageHistory** obtiene historial usando `get_session_history(session_id)`
3. **Historial se inyecta** en el prompt usando `MessagesPlaceholder`
4. **Prompt completo** (system + history + input) → LLM
5. **Respuesta generada** → Parser
6. **Respuesta guardada** en el historial automáticamente

### Componentes Clave:

- ✅ **ChatMessageHistory**: Almacena mensajes de una conversación
- ✅ **MessagesPlaceholder**: Placeholder en el prompt para el historial
- ✅ **RunnableWithMessageHistory**: Envuelve la cadena para añadir memoria
- ✅ **get_session_history**: Función que gestiona múltiples historiales

### Ventajas:

- 🔄 **Contexto persistente**: La IA recuerda conversaciones anteriores
- 🎯 **Conversaciones naturales**: Permite preguntas de seguimiento
- 🔀 **Múltiples sesiones**: Cada `session_id` tiene su propio historial
- 🛠️ **Flexible**: Puedes usar diferentes backends (memoria, Redis, DB)


## Notas Técnicas

### Limpiar Historial de una Sesión

```python
# Limpiar el historial de una sesión específica
store[session_id] = ChatMessageHistory()
```

### Persistencia en Producción

En producción, reemplaza el diccionario `store` con:

- **Redis**: Para alta velocidad y escalabilidad
- **PostgreSQL/MongoDB**: Para persistencia duradera
- **Memoria compartida**: Para aplicaciones multi-proceso

### Combinar Memoria con RAG

Puedes combinar memoria conversacional con RAG:

```python
rag_chain_with_memory = RunnableWithMessageHistory(
    rag_chain,  # Tu cadena RAG
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)
```

Esto permite que el sistema RAG recuerde el contexto de la conversación mientras responde basándose en documentos.


## Ejercicio Práctico

### Tu Turno

1. **Crea una nueva conversación** con un `session_id` diferente
2. **Haz una serie de preguntas** donde cada una depende de la anterior
3. **Verifica que la IA recuerda** el contexto completo

Ejemplo de flujo:
- Turno 1: "Soy un estudiante de ingeniería"
- Turno 2: "Estudio en la Universidad Nacional"
- Turno 3: "¿Dónde estudio?"
- Turno 4: "¿Qué carrera estudio?"

¡Experimenta con diferentes tipos de preguntas de seguimiento!


In [ ]:
# Tu código aquí
# Crea tu propia conversación con memoria

# session_ejercicio = "mi_conversacion"
# config_ejercicio = {"configurable": {"session_id": session_ejercicio}}
# 
# # Tu primera pregunta
# respuesta1 = chain_with_memory.invoke(
#     {"input": "Tu mensaje aquí"},
#     config=config_ejercicio
# )
# print(f"IA: {respuesta1}")

